In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 Generate ParamFree_5FoldCV_seed62.xlsx
 -- 5-Fold CV evaluation of the 8 parameter-free fusion rules on the 80%
    TRAIN folds, seed = 62. No command-line arguments: every setting is a
    constant below. Edit the constants and re-run to change the report.
================================================================================

Output columns: Fold | System | Rule | EER (%) | Rank-1 @ FAR=1% | N_train

USAGE
-----
    python generate_paramfree_seed62_report.py

================================================================================
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Callable, Dict, List

import numpy as np
import pandas as pd
from scipy.io import loadmat
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# ------------------------------------------------------------------------
# Reuse the normalization / fold-splitting / metric building blocks from
# the p-dependent evaluator, so both studies stay consistent with each
# other. evaluate_pdep_train_5fold.py must sit in the same folder as this
# script (or be importable on the Python path).
# ------------------------------------------------------------------------
sys.path.insert(0, str(Path.cwd()))

from evaluate_pdep_train_5fold import (  # noqa: E402
    DataConfig,
    SplitConfig,
    SystemDefinition,
    SYSTEM_DEFINITIONS,
    compute_eer,
    compute_rank1_at_far,
    is_valid,
    make_folds,
    make_normalizer,
    split_genuine_impostor,
    submatrix,
    train_indices_for_fold,
)


# ==============================================================================
#  CONFIGURATION -- every setting used to produce this report, in one place.
#  No argparse: change a value here and re-run the script.
# ==============================================================================

MAT_FILE = "870d37a7-673d-49ff-8a98-47b9b519237f.mat"
SEED = 62
N_FOLDS = 5
FAR_TARGET = 0.01
OUTPUT_DIR = "."
OUTPUT_FILENAME_TEMPLATE = f"Results/ParamFree_5FoldCV_seed{SEED}.xlsx"


# ==============================================================================
#  PARAMETER-FREE FUSION RULES
#  Each rule is f(x, y) -> fused score, no p argument.
# ==============================================================================

def rule1(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = xy * max(x,y)"""
    return x * y * np.maximum(x, y)


def rule2(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = xy * (x+y-xy)"""
    xy = x * y
    return xy * (x + y - xy)


def rule3(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = xy * min(x,y)"""
    return x * y * np.minimum(x, y)


def rule4(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = xy + (xy)^2 - xy*min(x,y)"""
    xy = x * y
    return xy + xy ** 2 - xy * np.minimum(x, y)


def rule5(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = (x+y-xy) + min(x,y) - (x+y-xy)*min(x,y)"""
    prob_sum = x + y - x * y
    mn = np.minimum(x, y)
    return prob_sum + mn - prob_sum * mn


def rule6(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = x+y - xy*(x+y-xy)"""
    xy = x * y
    return (x + y) - xy * (x + y - xy)


def rule7(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = (x+y-xy) + max(x,y) - (x+y-xy)*max(x,y)"""
    prob_sum = x + y - x * y
    mx = np.maximum(x, y)
    return prob_sum + mx - prob_sum * mx


def rule8(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """F = (x+y-xy) * max(x,y)"""
    prob_sum = x + y - x * y
    return prob_sum * np.maximum(x, y)


PARAMFREE_RULES: Dict[str, Callable[[np.ndarray, np.ndarray], np.ndarray]] = {
    "Rule1": rule1, "Rule2": rule2, "Rule3": rule3, "Rule4": rule4,
    "Rule5": rule5, "Rule6": rule6, "Rule7": rule7, "Rule8": rule8,
}
RULE_DISPLAY_ORDER: List[str] = list(PARAMFREE_RULES.keys())


def apply_paramfree_rule(rule_name: str, x: np.ndarray, y: np.ndarray,
                         invalid_value: float) -> np.ndarray:
    """Fuse two normalized matrices with a parameter-free rule (no p)."""
    valid = is_valid(x, invalid_value) & is_valid(y, invalid_value)
    x_safe = np.where(valid, x, 0.0)
    y_safe = np.where(valid, y, 0.0)

    with np.errstate(invalid="ignore", divide="ignore"):
        fused = PARAMFREE_RULES[rule_name](x_safe, y_safe)

    fused = np.clip(fused, 0.0, 1.0)
    return np.where(valid, fused, invalid_value)


def build_system_matrix_paramfree(system: SystemDefinition,
                                  normalized: Dict[str, np.ndarray],
                                  rule_name: str,
                                  invalid_value: float) -> np.ndarray:
    """Fuse the system's two modalities; cascade a third stage if defined."""
    mod_a, mod_b = system.stage1
    fused = apply_paramfree_rule(rule_name, normalized[mod_a], normalized[mod_b],
                                 invalid_value)
    if system.is_cascaded:
        fused = apply_paramfree_rule(rule_name, fused,
                                     normalized[system.stage2_modality],
                                     invalid_value)
    return fused


# ==============================================================================
#  STEP 1 -- RUN THE 5-FOLD SWEEP (no p dimension)
# ==============================================================================

def run_paramfree_sweep() -> pd.DataFrame:
    """
    Evaluate every (fold, system, rule) combination on that fold's TRAIN
    submatrix, using the constants defined above. Returns the long-format
    results table.
    """
    data_cfg = DataConfig(mat_file=MAT_FILE)
    split_cfg = SplitConfig(n_folds=N_FOLDS, seed=SEED, shuffle=True)

    contents = loadmat(MAT_FILE)
    raw_matrices = {
        modality: np.asarray(contents[key], dtype=float)
        for modality, key in data_cfg.matrix_keys.items()
    }
    n_subjects = next(iter(raw_matrices.values())).shape[0]
    folds = make_folds(n_subjects, split_cfg)

    records: List[dict] = []
    for fold_idx in range(N_FOLDS):
        train_ids = train_indices_for_fold(folds, fold_idx, n_subjects)

        normalized_train: Dict[str, np.ndarray] = {}
        for modality, matrix in raw_matrices.items():
            train_block = submatrix(matrix, train_ids)
            normalizer = make_normalizer(data_cfg, label=modality)
            normalized_train[modality] = normalizer.fit_transform(train_block)

        for system_name, system in SYSTEM_DEFINITIONS.items():
            for rule_name in RULE_DISPLAY_ORDER:
                fused = build_system_matrix_paramfree(
                    system, normalized_train, rule_name, data_cfg.invalid_value
                )
                genuine, impostor = split_genuine_impostor(fused, data_cfg.invalid_value)

                records.append({
                    "Fold": fold_idx + 1,
                    "System": system_name,
                    "Rule": rule_name,
                    "EER (%)": round(compute_eer(genuine, impostor) * 100, 4),
                    "Rank-1 @ FAR=1%": round(
                        compute_rank1_at_far(fused, FAR_TARGET, data_cfg.invalid_value) * 100,
                        4,
                    ),
                    "N_train": len(train_ids),
                })

        print(f"  Fold {fold_idx + 1}/{N_FOLDS} done ({len(train_ids)} train subjects).")

    report = pd.DataFrame.from_records(records)
    system_order = list(SYSTEM_DEFINITIONS.keys())
    report["System"] = pd.Categorical(report["System"], system_order, ordered=True)
    report["Rule"] = pd.Categorical(report["Rule"], RULE_DISPLAY_ORDER, ordered=True)
    return report.sort_values(["System", "Rule", "Fold"]).reset_index(drop=True)


# ==============================================================================
#  STEP 2 -- WRITE THE FORMATTED, SEED-TAGGED XLSX
# ==============================================================================

def write_report(report: pd.DataFrame) -> Path:
    font_name = "Arial"
    theme_color = "2E7D32"

    header_fill = PatternFill("solid", fgColor=theme_color)
    header_font = Font(name=font_name, bold=True, color="FFFFFF", size=10)
    title_font = Font(name=font_name, bold=True, size=13, color=theme_color)
    subtitle_font = Font(name=font_name, italic=True, size=9, color="555555")
    cell_font = Font(name=font_name, size=10)
    border = Border(*(Side(style="thin", color="D9D9D9") for _ in range(4)))

    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = "Results"

    n_systems = report["System"].nunique()
    n_rules = report["Rule"].nunique()

    worksheet.cell(row=1, column=1,
                   value=f"Parameter-Free Fusion Rules \u2014 5-Fold CV on 80% TRAIN "
                         f"(seed={SEED})").font = title_font
    worksheet.cell(row=2, column=1,
                   value=f"{n_systems} systems x {n_rules} parameter-free rules x "
                         f"{N_FOLDS} disjoint folds = {len(report)} rows. No p to sweep "
                         "-- each rule is a single fixed formula. Each fold's TRAIN "
                         "subset (~80%) is normalized independently (min-max fit on "
                         "that fold's train subjects only).").font = subtitle_font

    header_row = 4
    for col_idx, column in enumerate(report.columns, start=1):
        cell = worksheet.cell(row=header_row, column=col_idx, value=column)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    for row_offset, record in enumerate(report.itertuples(index=False), start=1):
        for col_idx, value in enumerate(record, start=1):
            column = report.columns[col_idx - 1]
            cell = worksheet.cell(row=header_row + row_offset, column=col_idx, value=value)
            cell.font = cell_font
            cell.border = border
            if column in ("EER (%)", "Rank-1 @ FAR=1%"):
                cell.number_format = "0.0000"

    worksheet.freeze_panes = worksheet.cell(row=header_row + 1, column=1)
    _autofit_columns(worksheet, len(report.columns), header_row, len(report))

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / OUTPUT_FILENAME_TEMPLATE.format(seed=SEED)
    workbook.save(output_path)
    return output_path


def _autofit_columns(worksheet, n_columns: int, header_row: int,
                     n_data_rows: int, minimum: int = 10, maximum: int = 28) -> None:
    for col_idx in range(1, n_columns + 1):
        letter = get_column_letter(col_idx)
        longest = max(
            (len(str(worksheet.cell(row=r, column=col_idx).value))
             for r in range(header_row, header_row + n_data_rows + 1)
             if worksheet.cell(row=r, column=col_idx).value is not None),
            default=minimum,
        )
        worksheet.column_dimensions[letter].width = min(max(longest + 2, minimum), maximum)


# ==============================================================================
#  ENTRY POINT -- no argparse, just run it.
# ==============================================================================

if __name__ == "__main__":
    print(f"Running 5-fold parameter-free sweep: seed={SEED}, n_folds={N_FOLDS} ...")
    report = run_paramfree_sweep()
    print(f"Sweep complete: {len(report)} rows.")

    output_path = write_report(report)
    print(f"\nReport written to: {output_path}")
    print(report.head(10).to_string(index=False))




Running 5-fold parameter-free sweep: seed=62, n_folds=5 ...
  Fold 1/5 done (413 train subjects).
  Fold 2/5 done (413 train subjects).
  Fold 3/5 done (414 train subjects).
  Fold 4/5 done (414 train subjects).
  Fold 5/5 done (414 train subjects).
Sweep complete: 320 rows.

Report written to: Results\ParamFree_5FoldCV_seed62.xlsx
 Fold System  Rule  EER (%)  Rank-1 @ FAR=1%  N_train
    1     S1 Rule1   4.8190          88.5922      413
    2     S1 Rule1   3.6643          91.2621      413
    3     S1 Rule1   3.6232          90.0966      414
    4     S1 Rule1   4.3754          89.8305      414
    5     S1 Rule1   4.6005          91.0412      414
    1     S1 Rule2   4.1262          88.5922      413
    2     S1 Rule2   3.6361          91.0194      413
    3     S1 Rule2   3.3816          90.0966      414
    4     S1 Rule2   3.8741          90.3148      414
    5     S1 Rule2   4.3584          90.7990      414
